In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 12 samples and 14441 columns.
First few rows of the data:


,os,os event,pfs,pfs event,sample_title,tissue,treatment,overall response,age,biomaterial provider,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,6.505133,1,2.004107,1,R009,breast,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Progressive Disease,49,"Alberto J. Montero,alberto.montero@uhhospitals...",...,7.886025,5.469030,7.033145,8.653241,4.809948,5.473786,7.548196,6.930863,6.056256,7.738258
1,41.330595,1,8.344969,1,R010_PRE,unspecified metastatic site,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,62,"Alberto J. Montero,alberto.montero@uhhospitals...",...,6.810587,4.255260,5.839894,6.943901,3.219846,4.129328,6.085884,6.036035,4.764023,6.600902
2,11.039014,1,11.039014,0,R012,lymph node,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,70,"Alberto J. Montero,alberto.montero@uhhospitals...",...,7.459401,5.454017,7.141900,7.864083,5.298725,5.802623,7.851647,7.127841,5.987570,8.080737
3,4.928131,1,4.928131,0,R013,lung,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Partial Response,45,"Alberto J. Montero,alberto.montero@uhhospitals...",...,7.354993,4.727607,6.573172,7.545722,4.605355,5.074363,7.311338,6.425795,5.911971,6.762733
4,18.759754,1,1.905544,1,R014_PRE,sternum,"Carboplatin, Nab-Paclitaxel, and Pembrolizumab",Progressive Disease,36,"Alberto J. Montero,alberto.montero@uhhospitals...",...,7.486330,5.415407,7.048296,7.943261,4.677227,5.441739,7.530151,7.112542,6.053870,7.572538


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Number of samples with incomplete data for os or os event: 0
Number of samples with complete data for both variables: 12
Number of censored samples (event_var == 0.0): 3
Number of events (event_var == 1.0): 9
Censored ratio (among complete): 3 / 12 = 0.250 (25.0%)
Number of samples with incomplete data for pfs or pfs event: 0
Number of samples with complete data for both variables: 12
Number of censored samples (event_var == 0.0): 8
Number of events (event_var == 1.0): 4
Censored ratio (among complete): 8 / 12 = 0.667 (66.7%)
Updated stats for 2 of 2 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE241876/description.json
